<a href="https://colab.research.google.com/github/mchadw/projSwing/blob/main/projSwing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
matches_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-matches.csv')
points_2020s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv')
points_2010s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2010s.csv')

points_raw = pd.concat([points_2010s_raw, points_2020s_raw])

matches_raw.head(10)
matches_raw.shape

/tmp/ipykernel_2908/1195214542.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  points_2020s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv')


(7566, 15)

In [ ]:
points_raw.head(10)

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
0,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,1,0,0,0,0.0,0-0,1,True,1,4n,4b1b1b3n@,NaN,1
1,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,2,0,0,0,0.0,15-0,1,True,1,5n,5f28f2b1b1b2f3f3d@,NaN,1
2,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,3,0,0,0,0.0,30-0,1,True,1,4C,NaN,NaN,1
3,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,4,0,0,0,0.0,40-0,1,True,1,4r2d#,NaN,NaN,1
4,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,5,0,0,1,0.0,0-0,2,True,2,5d,c4b1b1d!@,NaN,1
5,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,6,0,0,1,0.0,0-15,2,True,2,4n,4f38f2b1b1b1b1w@,NaN,1
6,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,7,0,0,1,0.0,0-30,2,True,2,6f#,NaN,NaN,2
7,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,8,0,0,1,0.0,15-30,2,True,2,4f38f+3f1i^1i3n#,NaN,NaN,2
8,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,9,0,0,1,0.0,30-30,2,True,2,4w,6f17f3f3f3n#,NaN,1
9,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,10,0,0,1,0.0,30-40,2,True,2,4r17b1m3o3*,NaN,NaN,2


In [ ]:
import random
import re

PLAYER = "Carlos Alcaraz"

MATCHES_PATH = "/content/drive/MyDrive/projSwing/charting-m-matches.csv"
POINTS_2020S_PATH = "/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv"
POINTS_2010S_PATH = "/content/drive/MyDrive/projSwing/charting-m-points-2010s.csv"

matches_raw = pd.read_csv(MATCHES_PATH)
points_2020s_raw = pd.read_csv(POINTS_2020S_PATH, low_memory=False)
points_2010s_raw = pd.read_csv(POINTS_2010S_PATH, low_memory=False)
points_raw = pd.concat([points_2010s_raw, points_2020s_raw], ignore_index=True)

p1_is_player = matches_raw["Player 1"].fillna("").str.contains(PLAYER, case=False, regex=False)
p2_is_player = matches_raw["Player 2"].fillna("").str.contains(PLAYER, case=False, regex=False)
# Keep matches where PLAYER is on exactly one side.
keep = p1_is_player ^ p2_is_player
player_matches = matches_raw.loc[keep, ["match_id", "Player 1", "Player 2"]].copy()
player_matches["player_side"] = 2
player_matches.loc[p1_is_player.loc[keep], "player_side"] = 1

match_meta = player_matches[["match_id", "player_side"]].drop_duplicates("match_id")
player_points = points_raw.merge(match_meta, on="match_id", how="inner")

print(f"{PLAYER}: {match_meta['match_id'].nunique()} matches, {len(player_points)} points")

FH = set("frvhjulo")
BH = set("bszikymp")
SHOT = re.compile(r"([fbrsvzopuylmhijktq])[+\-=;^]*([0-3])?[789]?[nwdxe!g]*[*#@C]?")
SERVE_ONLY = re.compile(r"^c*[0456]?\+?[nwdxegV!]+$")
SERVE_PREFIX = re.compile(r"^c*[0456]?\+?")
LABELS = ["f1", "f2", "f3", "b1", "b2", "b3"]


def rally(first, second):
    """Return (shot buckets, rally_starts_with_return).

    A bucket is f/b plus depth 1-3, or None when the token is not one of those six.
    The serve number is not a bucket. The first bucket is the returner's shot.
    """
    code = first if isinstance(first, str) else ""
    if SERVE_ONLY.fullmatch(code):
        code = second if isinstance(second, str) else ""
    code = SERVE_PREFIX.sub("", code)
    starts_with_return = True
    shots = []
    for stroke, depth in SHOT.findall(code):
        hand = "f" if stroke in FH else "b" if stroke in BH else None
        shots.append(hand + depth if hand and depth in ("1", "2", "3") else None)
    return shots, starts_with_return


def hitters_for(n_shots, server, starts_with_return):
    """Player index (1 or 2) for each parsed shot, alternating from the first hitter."""
    server = int(server)
    returner = 3 - server
    hitter = returner if starts_with_return else server
    out = []
    for _ in range(n_shots):
        out.append(hitter)
        hitter = 3 - hitter
    return out


def response_detail(first, second, svr, player_side):
    """Shots, hitter ids, and opponent->player bucket pairs for one point."""
    shots, starts_with_return = rally(first, second)
    try:
        server = int(svr)
        side = int(player_side)
    except (TypeError, ValueError):
        return shots, [], []
    if server not in (1, 2) or side not in (1, 2):
        return shots, [], []
    who = hitters_for(len(shots), server, starts_with_return)
    opponent_side = 3 - side
    pairs = []
    for i in range(len(shots) - 1):
        incoming, reply = shots[i], shots[i + 1]
        if incoming and reply and who[i] == opponent_side and who[i + 1] == side:
            pairs.append((incoming, reply))
    return shots, who, pairs


def build_response_matrix(points, n_examples=8, seed=0):
    """Count opponent shot -> player reply. Rows are the opponent bucket."""
    matrix = pd.DataFrame(0, index=LABELS, columns=LABELS)
    rng = random.Random(seed)
    stats = {"pairs": 0, "empty": 0, "with_pair": 0, "points": 0}
    examples = []
    for first, second, svr, side in zip(
        points["1st"], points["2nd"], points["Svr"], points["player_side"]
    ):
        stats["points"] += 1
        shots, who, pairs = response_detail(first, second, svr, side)
        if not shots:
            stats["empty"] += 1
        if pairs:
            stats["with_pair"] += 1
            for incoming, reply in pairs:
                matrix.loc[incoming, reply] += 1
                stats["pairs"] += 1
            record = {
                "1st": first,
                "2nd": second,
                "Svr": svr,
                "player_side": side,
                "shots": shots,
                "hitters": who,
                "counted": [f"{a}->{b}" for a, b in pairs],
            }
            if len(examples) < n_examples:
                examples.append(record)
            else:
                j = rng.randint(0, stats["with_pair"] - 1)
                if j < n_examples:
                    examples[j] = record
    matrix.index.name = "opponent_shot"
    matrix.columns.name = "player_response"
    return matrix, stats, examples


def _check_parser():
    shots, who, pairs = response_detail("4n", "4b1f2n@", 1, 1)
    assert shots == ["b1", "f2"], shots
    assert who == [2, 1], who
    assert pairs == [("b1", "f2")], pairs

    shots, who, pairs = response_detail("4n", "4f1b2f3", 1, 2)
    assert shots == ["f1", "b2", "f3"], shots
    assert who == [2, 1, 2], who
    assert pairs == [("b2", "f3")], pairs

    shots, who, pairs = response_detail("4C", None, 1, 1)
    assert shots == [] and pairs == []

    shots, who, pairs = response_detail("6f1b2*", None, 1, 1)
    assert shots == ["f1", "b2"], shots
    assert who == [2, 1], who
    assert pairs == [("f1", "b2")], pairs

    shots, who, pairs = response_detail("6f1f0b2", None, 2, 1)
    assert shots[0] == "f1" and shots[1] is None and shots[2] == "b2"
    assert pairs == []


_check_parser()
print("parser and hitter checks passed")

matrix, stats, examples = build_response_matrix(player_points)
n = stats["points"] or 1
print(f"matches: {player_points['match_id'].nunique()}")
print(f"points: {stats['points']}")
print(f"opponent -> {PLAYER} pairs: {stats['pairs']}")
print(f"points with an empty shot list: {stats['empty'] / n:.1%}")
print(f"points with at least one counted pair: {stats['with_pair'] / n:.1%}")
print("opponent-shot counts (row sums):")
print(matrix.sum(axis=1).astype(int))
low = matrix.sum(axis=1)
print("rows with fewer than 30 opponent shots:", list(low[low < 30].index))
print(matrix)

print(pd.DataFrame(examples).to_string())

# A row of NaNs means that opponent bucket had zero shots. It is not a uniform distribution.
row_sums = matrix.sum(axis=1)
prob_matrix = matrix.div(row_sums, axis=0)
print(prob_matrix)

Carlos Alcaraz: 222 matches, 36949 points
parser and hitter checks passed
matches: 221
points: 36949
opponent -> Carlos Alcaraz pairs: 50593
points with an empty shot list: 13.5%
points with at least one counted pair: 59.9%
opponent-shot counts (row sums):
opponent_shot
f1     8101
f2     7712
f3     9450
b1     3887
b2     9081
b3    12362
dtype: int64
rows with fewer than 30 opponent shots: []
player_response    f1    f2    f3    b1    b2    b3
opponent_shot                                      
f1               3345  2232  2501     7     6    10
f2               2058  1105  2261   365   719  1204
f3                324   121   386  1491  2330  4798
b1               1902   806  1163     6     4     6
b2               3314  1111  2605   509   707   835
b3               1137   316  1550  2044  2167  5148
                       1st                     2nd  Svr  player_side                                 shots                      hitters                           counted
0          6b38